## Prerequisites
 **LOCAL** Cytoscape (3.10.0 or greater) + **Local** Jupyter environment (including py4cytoscape).

### Setup required 

- Install Cytoscape on your PC. (See https://cytoscape.org/download.html)
- Install Jupyter on your PC. (See https://jupyter.org/install)
- Install py4cytoscape with `pip install py4cytoscape` on your PC.

## Getting started
**NOTE: To run this notebook, you must manually start Cytoscape first – don’t proceed until you have started Cytoscape.**

First, confirm that you have everything installed and running:

In [1]:
### necessary imports
import py4cytoscape as p4c
import os
from os.path import join, exists, dirname
import pandas as pd
from itertools import chain
import time
from datetime import date
import numpy as np
from collections import defaultdict
import re
import pickle
from pyoma.browser import db
import pyoma.browser.models as db_models
import seaborn as sns
### for showing examples and distribution figures
from IPython.display import Image, display
import matplotlib.pyplot as plt
from pprint import pprint

print("This notebook was tested with Cytoscape version 3.10.4 and py4cytoscape version 1.12.0\n")
### checking that cytoscape is running - this should print out 'You are connected to Cytoscape!' 
### and the version of Cytoscape you are connected to
p4c.cytoscape_ping()
p4c.cytoscape_version_info()


This notebook was tested with Cytoscape version 3.10.4 and py4cytoscape version 1.12.0

You are connected to Cytoscape!


{'apiVersion': 'v1',
 'cytoscapeVersion': '3.10.4',
 'automationAPIVersion': '1.12.0',
 'py4cytoscapeVersion': '1.12.0'}

# HogProf results visualization
Start by specifying the input directory, that includes all files generated by HogProf, and the output directory, where new files will be generated

In [ ]:
### set input and output directories
currentdir = os.getcwd()
print("Current directory is: %s" %(currentdir))

### Replace with your own paths
inputdir = "."
outdir = inputdir

### make output folder if it doesn't exist already
if not exists(outdir):
    os.mkdir(outdir)
    print("Created folder: %s" %(outdir))

Current directory is: /home/agavriil/Documents/venom_project/venom_scripts


## Visualisation styles
Importing premade styles from file. These are used in the examples below. Any custom styles would need to be imported first and then called in the relevant functions

In [ ]:
### import styles file - this is for later
styles_file = ''
### when we have some styles we like we can save them here and import them every time
styles_dict = {}
### keep track of why we have each style her
styles_description_dict = {}
### check if all the custom styles are already imported in Cytoscape
styles = p4c.get_visual_style_names()
if not all([style in styles for style in styles_dict.values()]):
    print("Importing visual styles")
    p4c.import_visual_styles(styles_file)

### checking layout names and tweakable attributes - uncomment one at a time
#p4c.get_layout_names()
#p4c.get_layout_property_names('attribute-circle')
#p4c.get_layout_property_names('force-directed')
#p4c.get_layout_name_mapping()

### Tweakable visual properties

In [4]:
# The colours are defined by hexadecimal codes. You can use a colour picker tool to find the code for the colour you want (e.g. http://medialab.github.io/iwanthue/)

# Commonly used node visual properties in Cytoscape
common_node_properties = [
    "NODE_FILL_COLOR",         # Color of the node
    "NODE_SHAPE",              # Shape of the node (DIAMOND, ELLIPSE, HEXAGON, OCTAGON, PARALLELOGRAM, RECTANGLE, ROUND_RECTANGLE, TRIANGLE, VEE)
    "NODE_SIZE",               # Size of the node (scales width and height together) (> 0)
    "NODE_LABEL",              # Text label for the node (default is the node name) 
    "NODE_LABEL_COLOR",        # Color of the node label
    "NODE_LABEL_FONT_SIZE",    # Font size of the node label (> 1)
    "NODE_BORDER_WIDTH",       # Width of the node's border (>= 0)
    "NODE_BORDER_COLOR",       # Color of the node's border
    "NODE_TRANSPARENCY",       # Transparency of the node (0-255)
    "NODE_LABEL_TRANSPARENCY", # Transparency of the node label (0-255)
]
print("Commonly used node properties:")
pprint(common_node_properties)

# Commonly used edge visual properties in Cytoscape
# Define a dictionary of commonly used edge properties with brief descriptions
common_edge_properties = [
    "EDGE_WIDTH",               # Width of the edge (> 0)
    "EDGE_COLOR",               # Color of the edge
    "EDGE_TRANSPARENCY",        # Transparency of the edge (0-255 where 0 is fully transparent)
    "EDGE_LABEL",               # Text label for the edge, empty by default
    "EDGE_LABEL_COLOR",         # Color of the edge label
    "EDGE_LABEL_FONT_SIZE",     # Font size of the edge label (> 1)
    "EDGE_LINE_TYPE",           # Line type (SOLID, DASHED, DOTTED, ZIGZAG, PARALLEL_LINES, SEPARATE_PARALLEL_LINES)
    "EDGE_SOURCE_ARROW_SHAPE",  # Arrow shape at the source end (NONE, ARROW, DIAMOND, CIRCLE, T, DELTA, HALF_ARROW_TOP, HALF_ARROW_BOTTOM)
    "EDGE_TARGET_ARROW_SHAPE",  # Arrow shape at the target end (NONE, ARROW, DIAMOND, CIRCLE, T, DELTA, HALF_ARROW_TOP, HALF_ARROW_BOTTOM)
    "EDGE_SOURCE_ARROW_COLOR",  # Arrow color at the source end
    "EDGE_TARGET_ARROW_COLOR"  # Arrow color at the target end
]
print("\nCommonly used edge properties:")
pprint(common_edge_properties)

# Get help on the map_visual_property function to see how to use it
#help(p4c.map_visual_property)

### Here is a list of all visual properties that can be modified. You can also look for specific properties for nodes, edges, and networks
#visual_properties = p4c.get_visual_property_names()
#node_properties = [prop for prop in visual_properties if 'NODE' in prop]
#print("\nAvailable node properties:")
#pprint(node_properties)
try:
    ### import network example in order to get the current visual style
    p4c.open_session(join(currentdir, 'network_mini_example.cys'))
    style_details = p4c.get_current_style()     # Get the current visual style - only works if a network is loaded
    print(f"\nDetails for Visual Style default:")
    print(style_details)
except:
    print("\nNo network loaded. Please load a network to get the current visual style.")

Commonly used node properties:
['NODE_FILL_COLOR',
 'NODE_SHAPE',
 'NODE_SIZE',
 'NODE_LABEL',
 'NODE_LABEL_COLOR',
 'NODE_LABEL_FONT_SIZE',
 'NODE_BORDER_WIDTH',
 'NODE_BORDER_COLOR',
 'NODE_TRANSPARENCY',
 'NODE_LABEL_TRANSPARENCY']

Commonly used edge properties:
['EDGE_WIDTH',
 'EDGE_COLOR',
 'EDGE_TRANSPARENCY',
 'EDGE_LABEL',
 'EDGE_LABEL_COLOR',
 'EDGE_LABEL_FONT_SIZE',
 'EDGE_LINE_TYPE',
 'EDGE_SOURCE_ARROW_SHAPE',
 'EDGE_TARGET_ARROW_SHAPE',
 'EDGE_SOURCE_ARROW_COLOR',
 'EDGE_TARGET_ARROW_COLOR']
Opening /home/agavriil/Documents/venom_project/venom_scripts/network_mini_example.cys...

Details for Visual Style default:
default


### Custom style

In [6]:
###### Creating a custom style programmatically (styles can also be edited manually in the Cytoscape GUI)

### Choose a name for the style and if you want to save it (will overwrite existing styles file with the same name)
style_name = "Custom"
style_file = join(currentdir, f'{style_name}_style.xml')
save_style = False


def rgb_to_hex(r, g, b):
  # Convert 0-1 values to 0-255 integers
  r = int(r * 255)
  g = int(g * 255) 
  b = int(b * 255)
  return f"#{('{:02X}' * 3).format(r, g, b)}"

def create_custom_style(style_name, node_colour_col, node_label_col, edge_width_col, node_size_col, node_colour='discreet', node_colour_values = [], 
                        node_colour_attrs=[], node_size_max=70, max_hits=100
                        ):
    print("Note: creating a style requirs the columns to be present in the currently open network")
    ### Delete the style if it already exists, to avoid multiple styles with the same name e.g. 'Custom-1', 'Custom-2' etc.
    try:
        p4c.delete_visual_style(style_name)
        print(f"Deleted style {style_name}")
    except:
        print()
    ### Change default values for visual properties. Anything not specified in the mappings below will use these defaults
    # Here we define the default node to be filled with a light blue colour, to be an ellipse shape, to have a size of 30, and to have black labels
    # The default edge is defined to have a width of 2,to be a light grey colour and be about 40% transparent
    defaults = {
        "NODE_FILL_COLOR": "#6BAED6",  # Default fill color for nodes
        "NODE_SHAPE": "ELLIPSE",        # Default shape for nodes
        "NODE_SIZE": 35,                # Default size for nodes
        "NODE_LABEL_COLOR": "#000000",   # Default label color for nodes
        "NODE_BORDER_WIDTH": 0,         # Default border width for nodes
        "EDGE_WIDTH": 2,                # Default width for edges
        "EDGE_COLOR": "#999999",        # Default color for edges
        "EDGE_TRANSPARENCY": 150,        # Default transparency for edges
        "NETWORK_BACKGROUND_PAINT": "#FFFFFF"} # The default network is defined to have a white background

    ### Define mappings for node and edge visual properties. Here you choose the columns in your metadata table that will define these visual properties
    
    # The node color will be defined by the 'isgene' column, with the value 'False' being mapped to a pink colour - this uses discrete mapping, so the user needs to specify the colour for each value
    # Note: in discrete mapping, the values in the attribute list and the mappings list must be in the same order
    if node_colour == 'discreet':
        node_mapping = 'd'
    elif node_colour == 'continuous':
        node_mapping = 'c'
    node_color_mapping = p4c.map_visual_property(
        "NODE_FILL_COLOR",     # The visual property to map
        node_colour_col,              # The attribute column in the node table (if it is categorical strings you need discreet mapping)
        node_mapping,                   # 'd' stands for discreet mapping   / 'c' stands for continuous
        node_colour_values,             # List of attribute values to map   / numeric range for this variable
        node_colour_attrs            # Corresponding color for each attribute value / colors to interpolate between
    )

    # node size, for making e.g. the source nodes bigger
    node_size_mapping = p4c.map_visual_property(
        "NODE_SIZE",        # The visual property to map
        node_size_col,      # The attribute column in the node table
        'c',                # 'c' stands for continuous mapping
        [0, max_hits],      # Range of values
        [30,100]            # Corresponding size for each attribute value
    )

    # The node label will be defined by the 'name' column - this uses passthrough mapping, so the value in the column will be mapped directly to the visual property
    node_label_mapping = p4c.map_visual_property(
        "NODE_LABEL",   # The visual property to map
        node_label_col,         # The attribute column in the node table
        "p"             # 'p' stands for passthrough mapping
    )
    if edge_width_col != "interaction":
        # The edge width will be defined by the 'interaction' column, with the value 'dissociation' being mapped to a width of 20. 
        edge_width_mapping = p4c.map_visual_property(
            "EDGE_WIDTH",             # The visual property to map
            edge_width_col,            # The attribute column in the edge table
            "c",                      # 'c' stands for continuous mapping
            [0,1],                      # numeric range for this variable
            [1,10]                      # Rang of widths 
        )
    else: ### just for the example
        edge_width_mapping = p4c.map_visual_property(
        "EDGE_WIDTH",             # The visual property to map
        edge_width_col,            # The attribute column in the edge table
        "d",                      # 'd' stands for discrete mapping
        ["dissociation", "weakassociation"],         # List of attribute values to map
        [20,1]                      # Corresponding widths for each attribute value
    )
    # The edge color will be defined by the 'interaction' column, with the value 'dissociation' being mapped to a red colour. 
    edge_color_mapping = p4c.map_visual_property(
        "EDGE_COLOR",              # The visual property to map
        "interaction",             # The attribute column in the edge table
        "d",                       # 'd' stands for discrete mapping
        [],          # List of attribute values to map
        []                # Corresponding colors for each attribute value
    )
    # The transparency of the edge will be defined by the 'GeneForce' column. This uses continuous mapping, so the user needs to specify the range of values
    # Here a GeneForce value of 1 will be mapped to a transparency of 100, and a GeneForce value of 50 will be mapped to a transparency of 255
    # All values in between will be linearly interpolated
    edge_transparency_mapping = p4c.map_visual_property(
        "EDGE_TRANSPARENCY",        # The visual property to map
        "interaction",                  # The attribute column name in the edge table
        "d",                        # 'd' stands for discreet mapping
        [],                    # The points where mapping changes occur (thresholds)
        []                  # Corresponding transparency values at those points
    )

    # Gather mappings together in a list
    mappings = [node_color_mapping, node_label_mapping, node_size_mapping, edge_width_mapping, edge_color_mapping, edge_transparency_mapping]
    #print(style_name, defaults, mappings)

    # Construct the visual style object
    p4c.create_visual_style(style_name=style_name, defaults=defaults, mappings=mappings)
    print("Created style %s" %(style_name))



style_name = 'example_style'
node_colour_col = 'isgene'
node_label_col = 'name'
node_colour='discreet'
node_colour_values = [False] ##
node_colour_attrs=["#E7298A"] ##
edge_width_col = "interaction"
node_size_col = "isgene"
### Only create the style if it doesn't already exist
if style_name not in p4c.get_visual_style_names():
    create_custom_style(style_name, node_colour_col, node_label_col, edge_width_col, node_size_col, 
                        node_colour='discreet', node_colour_values = [], node_colour_attrs=[])

    # Apply the style to the network
    p4c.set_visual_style(style_name)

    # Save the style if desired
    if save_style:
        p4c.export_visual_styles(style_file)
        print(f"Saved style to {style_file}")


Note: creating a style requirs the columns to be present in the currently open network



In cyrest_delete(): Could not find Visual Style: example_style
In cyrest_post(): Could not create new Visual Style.


CyError: In cyrest_post(): Could not create new Visual Style.

# Generation of networks in Cytoscape

### 0: Importing information & preparing data for network visualization
The following code block imports all information necessary from the HogProf results files and prepares the data in a format ready to be visualized. No network is generated in cytoscape yet.

In [7]:
### simple version

### we need an edges df for cytoscape import
overwrite=False
cytoscape_input_file = os.path.join(inputdir, 'extracted_hits_from_subhogs.csv')
### get info from cytoscape_input file
edges_df = pd.read_csv(cytoscape_input_file)
### we need the source and target to be clear
rename_dict = {'query_hog':'source','target_hog':'target'}
edges_df = edges_df.rename(columns=rename_dict)
### taxname root_Metazoa needs to be Metazoa to match the metadata table
edges_df['taxname'] = edges_df['taxname'].replace('root_Metazoa', 'Metazoa')
edges_df.head()


,source,target,jaccard,empirical_t,taxname,same_fam
0,216_HOG:0006194_148,216_HOG:0028041_148,1.0,0.933594,Toxicofera,False
1,151_HOG:0009799_146,151_HOG:0071181_146,1.0,0.941406,Lepidosauria,False
2,151_HOG:0009799_146,151_HOG:0062007.2b_146,1.0,0.941406,Lepidosauria,False
3,151_HOG:0009799_146,151_HOG:0163148_146,1.0,0.941406,Lepidosauria,False
4,151_HOG:0009799_146,151_HOG:0044970.26a_146,1.0,0.941406,Lepidosauria,False


In [8]:
### we also need a nodes df, for all the metadata
hogs_metadata_file = "/home/agavriil/Documents/venom_project/A_venom_analysis_tidy/3_results/2_fastoma_results/levels_fastoma_metazoa_251114_subhogs_eventlim1_260619_queries/venomhog_full_subhog_metadata_summary.tsv"
nodes_df = pd.read_csv(hogs_metadata_file, sep='\t', index_col=0)
# create a column with venom, expr or "" depending on all_hits and all_hits_expr
nodes_df["nodetype"] = np.select(
    [ pd.to_numeric(nodes_df["all_hits"], errors="coerce").fillna(0) > 0,
     pd.to_numeric(nodes_df["all_hits_expr"], errors="coerce").fillna(0) > 0,],
    ["venom","expr",],default="")
print(len(nodes_df[nodes_df['nodetype']=='venom']))
print(len(nodes_df[nodes_df['nodetype']=='expr']))
print(len(nodes_df[nodes_df['nodetype']=='']))
nodes_df.head()

1363
7943
0


,is_toxin,all_hits,direct_hit,all_hits_expr,direct_hit_expr,level_taxon,total_species,venomous_species_percent,completeness_score,hashid,is_roothog,Protein families,nodetype,exprmods
subhog_id,,,,,,,,,,,,,,
0_HOG:0001226_1,NaN,0,0,8.0,1.0,Metazoa,327,0.119266,0.9534,579204,NaN,NaN,expr,"mod13,mod24,mod40"
1_HOG:0001226_2,NaN,0,0,8.0,1.0,Eumetazoa,326,0.119632,0.9532,579205,NaN,NaN,expr,"mod13,mod24,mod40"
3_HOG:0001226_3,NaN,0,0,8.0,1.0,Bilateria,320,0.109375,0.9524,579206,NaN,NaN,expr,"mod13,mod24,mod40"
8_HOG:0001226_312,NaN,0,0,6.0,1.0,Protostomia,149,0.134228,0.9255,579207,NaN,NaN,expr,"mod13,mod24,mod40"
13_HOG:0001226_313,NaN,0,0,6.0,1.0,Ecdysozoa,122,0.139344,0.9104,579208,NaN,NaN,expr,"mod13,mod24,mod40"


In [9]:
def adapt_to_cytoscape(nodes_df, edges_df, verbose=False):
    ### sanity check, everything in edges_df should be in nodes_df
    edge_ids = set(edges_df["source"]) | set(edges_df["target"])
    node_ids = set(nodes_df.index)

    # --- Check edges -> nodes ---
    # IDs referenced in edges but missing from nodes
    missing_in_nodes = edge_ids - node_ids

    ### nodes_df originally has only the queries but we should add empty rows for the hits 
    if len(missing_in_nodes)> 0:
        print(f"Adding {len(missing_in_nodes)} missing nodes")
        # create empty rows with same columns as nodes_df
        missing_df = pd.DataFrame(index=list(missing_in_nodes), columns=nodes_df.columns)
        # recover taxname from edges_df
        # each missing node may appear either as source or target
        taxname_map = {}
        for _, row in edges_df.iterrows():
            if row["source"] in missing_in_nodes:
                taxname_map[row["source"]] = row["taxname"]
            if row["target"] in missing_in_nodes:
                taxname_map[row["target"]] = row["taxname"]
        # assign recovered taxnames
        missing_df["level_taxon"] = missing_df.index.map(taxname_map)
        # append to nodes_df
        nodes_df = pd.concat([nodes_df, missing_df])

    # --- Check nodes -> edges ---
    # IDs present in nodes but never used in edges -> those are the queries without significant hits
    unused_nodes = node_ids - edge_ids
    if len(unused_nodes)>0:
        if verbose:
            print(f"Removing unused nodes: {len(unused_nodes)}")
        unused_nodes_df = nodes_df.loc[list(unused_nodes)].copy()
        # remove them from nodes_df
        nodes_df = nodes_df.drop(index=list(unused_nodes))
        # optional
        nodes_df = nodes_df.sort_index()
        unused_nodes_df = unused_nodes_df.sort_index()

    ### loud checks
    edge_ids = set(edges_df["source"]) | set(edges_df["target"])
    node_ids = set(nodes_df.index)
    all_edges_exist = edge_ids.issubset(node_ids)
    all_nodes_used = node_ids.issubset(edge_ids)
    if verbose:
        print("All edge IDs exist in nodes:", all_edges_exist)
        print("All nodes are used in edges:", all_nodes_used)

    ### fixes to be compatible with cytoscape
    ### cytoscape requires the nodes_df to have an 'id' column
    nodes_df["id"] = nodes_df.index
    # replace inf/-inf if present
    nodes_df = nodes_df.replace([np.inf, -np.inf], np.nan)
    edges_df = edges_df.replace([np.inf, -np.inf], np.nan)
    # add zero values where appropriate
    for col in ['all_hits', 'direct_hit','all_hits_expr','direct_hit_expr']:
        try:
            nodes_df[col] = nodes_df[col].fillna(0)
        except:
            print('Colunn not found:', col)
    # Cytoscape/JSON dislikes NaN
    nodes_df = nodes_df.fillna("")
    edges_df = edges_df.fillna("")
    # ensure edge ids are strings
    edges_df["source"] = edges_df["source"].astype(str)
    edges_df["target"] = edges_df["target"].astype(str)
    return nodes_df, edges_df

nodes_df, edges_df = adapt_to_cytoscape(nodes_df, edges_df)
nodes_df.head()

Adding 21075 missing nodes


,is_toxin,all_hits,direct_hit,all_hits_expr,direct_hit_expr,level_taxon,total_species,venomous_species_percent,completeness_score,hashid,is_roothog,Protein families,nodetype,exprmods,id
0_HOG:0026692_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0026692_1
0_HOG:0038713_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0038713_1
0_HOG:0055331_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0055331_1
0_HOG:0057897_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0057897_1
0_HOG:0064348_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0064348_1


In [10]:
if False:
    ### we need an edges df and a nodes df separate for cytoscape import
    overwrite=False
    ### change file paths as needed
    #cytoscape_input_file = os.path.join(outdir, 'extracted_hits_from_allvenomexpr_subhogs.csv')
    #cytoscape_input_file = os.path.join(outdir, 'extracted_hits_from_allvenomexpr_subhogs_strictsignificant_withtaxa.csv')
    #cytoscape_input_file = os.path.join(inputdir, 'extracted_hits_from_promisingantvenom_blastphogs.csv')
    cytoscape_input_file = os.path.join(inputdir, 'extracted_hits_from_allvenom_blastphogs.csv')
    cytoscape_nodes_file = cytoscape_input_file.replace('.csv', 'cytonodestable.csv')
    idmapper_pickle_file = os.path.join(inputdir, 'idmapper.pkl')
    #oma_path = "/home/agavriil/Documents/venom_project/A_venom_analysis_tidy/0_data/OmaServer.h5"
    oma_path = ""
    if not exists(cytoscape_input_file):
        print(f"File not found. Check input directory:\n{cytoscape_input_file}")

    ### function to categorize columns by suffix, to separate Gene_1, Gene_2 columns from other columns
    def categorize_columns_by_prefix(columns):
        # Initialize containers
        prefix_groups = defaultdict(list)  # Store columns by their prefix
        coldict = {}  # Map full column names to base names
        
        # Process each column in the list
        for col in columns:
            if col.startswith('query_'):
                prefix_groups['query'].append(col)
                coldict[col] = col.replace('query_', '')
            elif col.startswith('hit_'):
                prefix_groups['hit'].append(col)
                coldict[col] = col.replace('hit_', '')
        
        # Get lists of query and hit columns
        queries = prefix_groups.get('query', [])
        hits = prefix_groups.get('hit', [])
        
        return queries, hits, coldict

    ### get subhog object from id
    def get_subhog_keyword(subhog_id, omadb):
        
        subhog_obj = omadb.get_hog(subhog_id)
        #print(subhog_obj)
        subhog_omaobj = db_models.HOG(omadb, subhog_obj)
        #print(subhog_omaobj)
        fam_num = subhog_obj[0]
        keyword = omadb.get_roothog_keywords(fam_num)
        #print([keyword])
        nonspecific_keywords = ['Derived by automated computational analysis using gene prediction method Gnomon', 'hypothetical protein']
        memberslist = subhog_omaobj.members
        if keyword not in nonspecific_keywords:
            found_keyword = True
        else:
            found_keyword = False
        if not found_keyword:
            prot_counter = 0
            for prot in memberslist:
                #print(prot.canonicalid, prot.description)
                #if prot.canonicalid == 'A0A8C3FUX7':
                #    print('description',prot.description)
                if prot.description not in nonspecific_keywords:
                    keyword = prot.description
                    found_keyword = True
                prot_counter +=1
                if prot_counter > 50:
                    found_keyword = True
                if found_keyword:
                    break
                xrefs = prot.xrefs
                ### look for protein name if exists as a key in the list of dicts
                for xref in xrefs:
                    if found_keyword:
                        break
                    if 'Protein Name' == xref['source']:
                        keyword = xref['xref']
                        if keyword not in nonspecific_keywords:
                            found_keyword = True
                            break
        ###
        if keyword in nonspecific_keywords:
            keyword = "unknown"
        ### simplify if possible
        else:
            #keyword = keyword.replace("; Derived by automated computational analysis using gene prediction method: Gnomon.", "")
            keyword =  keyword.split(";")[0]
            keyword = keyword.split("[Source:")[0]
        #print([keyword])
        return keyword

    def annotate_table_from_oma(cytoscape_df, idmapper_pickle_file, oma_path):
        ### now we can clean up a bit more. get a better HOG name
        cytoscape_df['subhog_name'] = cytoscape_df['subhogid'].str.extract(r'(HOG:E\d+(?:\.\d+[a-z])*(?:\.\d+)*)', expand=False)
        ### get hog name and taxon name from subhog name
        cytoscape_df['hog_name'] = cytoscape_df['subhog_name'].apply(lambda x: str(x).split('.')[0])
        ### taxonid is the level column already
        ### get idmapper to decode level column
        with open(idmapper_pickle_file, 'rb') as f:
            idmapper = pickle.load(f)
            #print(idmapper)
            print(f"\nLoaded idmapper with {len(idmapper)} entries")
            # Get unique taxon IDs from level column
            unique_taxids = cytoscape_df['level'].unique()
            ### lets make sure they are strings in the df
            unique_taxids = [str(tid) for tid in unique_taxids]
            print(f"Unique taxon IDs found: {len(unique_taxids)}")
            cytoscape_df['level'] = cytoscape_df['level'].astype(str)  # Ensure level is str for mapping
            ### idmapper has taxonname: taxonid but we need the reverse
            # Create mini reverse mapping only for needed taxon IDs
            mini_reverse_mapper = {v: k for k, v in idmapper.items() if v in unique_taxids}
            print(f"Mini reverse mapper created with {len(mini_reverse_mapper)} entries")
            # Map level (taxonid) to taxon name using the smaller dictionary
            cytoscape_df['taxon_name'] = cytoscape_df['level'].map(mini_reverse_mapper).fillna('Unknown')
        ### Now we also need to get some kind of description of function of the subHOG
        ### get oma 
        omadb = db.Database(oma_path)
        ### apply function to cytoscape_df
        cytoscape_df['subhog_keyword'] = cytoscape_df['hog_name'].apply(lambda x: get_subhog_keyword(x, omadb))
        return cytoscape_df

    ### get info from cytoscape_input file
    edges_df = pd.read_csv(cytoscape_input_file)
    if overwrite or not os.path.exists(cytoscape_nodes_file):
        print('Creating nodes table for cytoscape')
        ### check for any query_ hit_ columns 
        queries, hits, coldict = categorize_columns_by_prefix(edges_df.columns)
        ### get all columns with query_ hit_ - all node columns
        queries_df = edges_df[queries]
        #print(queries_df)
        hits_df = edges_df[hits]
        #print(hits_df)
        ### rename columns to remove query_ hit_ prefixes
        queries_df = queries_df.rename(columns=coldict)
        queries_df.drop_duplicates(inplace=True)
        queries_df['isquery'] = True
        hits_df = hits_df.rename(columns=coldict)
        hits_df.drop_duplicates(inplace=True)
        hits_df['isquery'] = False
        #overlap_nodes = set(queries_df['subhogid']).intersection(set(hits_df['subhogid']))
        # Print diagnostics before processing
        print("Number of unique queries:", len(queries_df['subhogid'].unique()))
        print("Number of unique hits:", len(hits_df['subhogid'].unique()))
        print("Total unique nodes expected:", 
            len(set(queries_df['subhogid']).union(set(hits_df['subhogid']))))
        nodes_df = pd.concat([queries_df, hits_df])
        #print(nodes_df)
        nodes_df.drop_duplicates(inplace=True, subset=['subhogid'])
        #print(nodes_df)
        # Print diagnostics after processing
        print("\nAfter processing:")
        print("Number of nodes after concat and dedup:", len(nodes_df))
        print("Number of nodes with isquery=True:", len(nodes_df[nodes_df['isquery']==True]))
        print("Number of nodes with isquery=False:", len(nodes_df[nodes_df['isquery']==False]))
        #print(nodes_df)
        ### run function to add columns
        if not oma_path == "":
            nodes_df = annotate_table_from_oma(nodes_df, idmapper_pickle_file, oma_path)
        nodes_df = nodes_df.rename(columns={'subhogid':'id'})
        ### save to table:
        nodes_df.to_csv(cytoscape_nodes_file)
    else:
        print("Reading nodes table for cytoscape from file")
        nodes_df = pd.read_csv(cytoscape_nodes_file, index_col=0)
        nodes_df = nodes_df.rename(columns={'subhogid':'id'})
    ### rename edges table so that source and target are clear
    rename_dict = {'query_subhogid':'source','hit_subhogid':'target'}
    edges_df = edges_df.rename(columns=rename_dict)
    ### replace all NaN in nodes table with ""
    nodes_df = nodes_df.fillna("")

    try:
        ### optional step to remove duplicates in level and subhog_name (will be redundant after HogProf is fixed)
        nodes_df = nodes_df.drop_duplicates(subset=['level', 'subhog_name'], keep='first')
    except Exception as e:
        print(f"Error removing duplicates: {e}")
    # Get list of remaining node IDs
    valid_node_ids = set(nodes_df['id'])
    # Filter edges to keep only those where both source and target are in valid_node_ids
    edges_df = edges_df[
        edges_df['source'].isin(valid_node_ids) & 
        edges_df['target'].isin(valid_node_ids)
    ]

    ### show that all is done
    print("\nDone")
    nodes_df
    #edges_df

    ##180mins, 106mins

## 0: Visualizing coevolution clusters 

The following code block will visualize the subHOGs and their coevolution edges

#### Warning: all currently open networks will be deleted before new ones are generated!

In [11]:
### simple version, just create a network (if it is a large network this will take a while)
max_edges_num = 80000
if edges_df.shape[0] < max_edges_num: 
    ### deleting existing networks to start over
    p4c.delete_all_networks()
    ### make coevolution network 
    collection_title = "HogProf results"
    p4c.create_network_from_data_frames(nodes_df, edges_df, title="venom HOGs coevolution network", collection=collection_title)
    p4c.layout_network('force-directed')
    today = date.today()
    todaystr = today.strftime("%y%m%d")
    ### save session for the future
    outcys_session_name = f'{todaystr}_HogProf_venom0.3medium_alltaxa_network.cys'
    print(f"Saving session in file {outcys_session_name}")
    p4c.save_session(os.path.join(outdir, outcys_session_name))
else:
    print("Network is too large. Consider visualizing slices.")

Applying default style...


In commands_post(): No network views selected.


Applying preferred layout
Saving session in file 260701_HogProf_venom0.3medium_alltaxa_network.cys


In [12]:
### helper functions
### visual style function
def apply_visual_style(style_name):
        try:
            print(f"Applying visual style: {style_name}")
            p4c.set_visual_style(style_name)
        except:
            print("Visual style not found. Using default.")
            p4c.set_visual_style(styles_dict['test'])
### save image function
def save_network_image(outputfile, filetype='JPEG'):
    try:
        p4c.export_image(outputfile, type=filetype)
        print(f"Saving figure in file {outputfile}")
    except:
        print("Error saving figure. File already exists?\n")

In [13]:
nodes_df.head()

,is_toxin,all_hits,direct_hit,all_hits_expr,direct_hit_expr,level_taxon,total_species,venomous_species_percent,completeness_score,hashid,is_roothog,Protein families,nodetype,exprmods,id
0_HOG:0026692_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0026692_1
0_HOG:0038713_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0038713_1
0_HOG:0055331_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0055331_1
0_HOG:0057897_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0057897_1
0_HOG:0064348_1,,0.0,0.0,0.0,0.0,Metazoa,,,,,,,,,0_HOG:0064348_1


In [14]:

### at this point we can create a custom style that includes a distinct colour for each taxon_name 
### creating a style requirs the columns to be present in the currently open network
style_name = 'hogprof_custom'
node_colour_col = 'level_taxon'     # column that defines the node colour
node_label_col = 'id'               # column that definde the node label
node_colour='discreet'              
edge_width_col = "jaccard"
node_size_col = "all_hits"          # column that definde the node width
max_hits = nodes_df["all_hits"].max()
if node_colour_col in nodes_df.columns and node_label_col in nodes_df.columns:
    node_colour_values = nodes_df[node_colour_col].unique()
    node_colour_attrs=sns.color_palette(None, len(node_colour_values))
    node_colour_attrs = [rgb_to_hex(*c) for c in node_colour_attrs]
    #print(node_colour_values)
    ### create it
    create_custom_style(style_name=style_name, node_colour_col=node_colour_col, node_label_col=node_label_col, node_size_col=node_size_col,
                        edge_width_col=edge_width_col, node_colour=node_colour, node_colour_values=node_colour_values,
                        node_colour_attrs=node_colour_attrs, node_size_max = max_hits)
else:
    ### create it
    create_custom_style(style_name=style_name, node_label_col='name', node_size_col=node_size_col, node_colour_col = 'is_toxin',
                        edge_width_col=edge_width_col, node_colour=node_colour)
    
### try it on open network
p4c.layout_network('force-directed')
try:
    apply_visual_style(style_name) ### using hogprof_custom from the previous block
except Exception as e:
    print(f"Error occurred while applying visual style: {e}")

Note: creating a style requirs the columns to be present in the currently open network



In cyrest_delete(): Could not find Visual Style: hogprof_custom


Created style hogprof_custom
Applying visual style: hogprof_custom


In [51]:
if False:
    ### deleting existing networks to start over
    p4c.delete_all_networks()
    ### check if all elements in nodes (id) are present in edges and vice versa
    def check_nodes_edges(nodes_df, edges_df):
        nodes = set(nodes_df['id'])
        edges = set(edges_df['source']).union(set(edges_df['target']))
        if nodes != edges:
            print("\nWARNING: Nodes and edges do not match!")
            print("Nodes not in edges:", nodes - edges)
            print("Edges not in nodes:", edges - nodes)
            print("Check step 0: importing of information!\n")
        else:
            print("Nodes and edges tables seem ok")
    #check_nodes_edges(nodes_df, edges_df)

    ### make coevolution network 
    collection_title = "HogProf results"
    p4c.create_network_from_data_frames(nodes_df, edges_df, title="venom HOGs coevolution network", collection=collection_title)
    p4c.layout_network('force-directed')
    
    ### and apply it
    apply_visual_style(style_name)
    ### special case for phage-defense


    ### give it a second for the style to be applied
    time.sleep(1)
    ### save the figure
    today = date.today()
    todaystr = today.strftime("%y%m%d")
    outfig_name=f'{todaystr}_HogProf_venom0.3medium_network.jpeg'
    # for more options see here: https://py4cytoscape.readthedocs.io/en/latest/reference/generated/py4cytoscape.network_views.export_image.html
    save_network_image(join(outdir, outfig_name), filetype='JPEG')
    ### save session for the future
    outcys_session_name = f'{todaystr}_HogProf_venom0.3medium_network.cys'
    print(f"Saving session in file {outcys_session_name}")
    p4c.save_session(os.path.join(outdir, outcys_session_name))



## 1: Taxon-based visualization
The following block will create one network visualization per taxonomic level
#### Warning: all currently open networks will be deleted before new ones are generated!

In [15]:
### at this point we can create a custom style that includes a distinct colour for venom and expression
style_name = 'hogprof_venom_expr_custom'
node_colour_col = 'nodetype'     # column that defines the node colour
node_label_col = 'id'               # column that definde the node label
node_colour='discreet'              
edge_width_col = "jaccard"
node_size_col = "all_hits"          # column that definde the node width
max_hits = nodes_df["all_hits"].max()
#node_colour_values = nodes_df[node_colour_col].unique()
node_colour_values = ["venom","expr",""]
node_colour_attrs=sns.color_palette(None, len(node_colour_values))
node_colour_attrs = [rgb_to_hex(*c) for c in node_colour_attrs]
#print(node_colour_values)



### deleting existing networks to start over - this took 2 mins for 180000 edges and 65000 nodes
p4c.delete_all_networks()
collection_title = "HogProf results per taxon"
edges_taxon_col = 'taxname'
nodes_taxon_col = 'level_taxon'
### separate the edges table into a list of dfs, per taxon_name of query
taxa_edges_dfs_list = []
first = True
for taxon_name in nodes_df[nodes_taxon_col].unique():
    taxon_nodes_df = nodes_df[nodes_df[nodes_taxon_col] == taxon_name]
    taxon_edges_df = edges_df[edges_df[edges_taxon_col] == taxon_name]
    print(f"Taxon {taxon_name} has {len(taxon_edges_df)} interactions among {len(taxon_nodes_df)} HOGs.")
    ### filter out possible hits that are in the nodes still - redundant thanks to our earlier check
    #all_nodes_in_edges = set(taxon_edges_df['source']).union(set(taxon_edges_df['target']))
    #taxon_nodes_df = nodes_df[nodes_df['id'].isin(all_nodes_in_edges)]
    taxon_nodes_df, taxon_edges_df = adapt_to_cytoscape(taxon_nodes_df, taxon_edges_df)
    if len(taxon_nodes_df) > 0:
        p4c.create_network_from_data_frames(taxon_nodes_df, taxon_edges_df, title=taxon_name, collection=collection_title)
        p4c.layout_network('force-directed')
        if first:
            ### create the new style
            create_custom_style(style_name=style_name, node_colour_col=node_colour_col, node_label_col=node_label_col, node_size_col=node_size_col,
                                edge_width_col=edge_width_col, node_colour=node_colour, node_colour_values=node_colour_values,
                                node_colour_attrs=node_colour_attrs, node_size_max = max_hits)
            first=False
        try:
            apply_visual_style(style_name) ### using hogprof_custom from the previous block
        except Exception as e:
            print(f"Error occurred while applying visual style: {e}")
        #break

    
### give it a second for the style to be applied
time.sleep(1)
### save the figure
today = date.today()
todaystr = today.strftime("%y%m%d")
### save session for the future
outcys_session_name = f'{todaystr}_HogProf_venom0.3medium_taxa_network.cys'
print(f"Saving session in file {outcys_session_name}")
p4c.save_session(os.path.join(outdir, outcys_session_name))

Taxon Metazoa has 21 interactions among 24 HOGs.


/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index


Applying default style...
Applying preferred layout
Note: creating a style requirs the columns to be present in the currently open network



In cyrest_delete(): Could not find Visual Style: hogprof_venom_expr_custom


Created style hogprof_venom_expr_custom
Applying visual style: hogprof_venom_expr_custom
Taxon Amniota has 3330 interactions among 1288 HOGs.


...

/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index


Applying default style...
Applying preferred layout
Applying visual style: hogprof_venom_expr_custom
Taxon Protostomia has 339 interactions among 332 HOGs.


/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index


Applying default style...
Applying preferred layout
Applying visual style: hogprof_venom_expr_custom
Saving session in file 260701_HogProf_venom0.3medium_taxa_network.cys


{}

In [ ]:
### filtered version - only keep all_hits >0 
nodes_filtered_df = nodes_df[ pd.to_numeric(nodes_df["all_hits"], errors="coerce") > 0]
edges_filtered_df = edges_df[edges_df['source'].isin(nodes_filtered_df.index)]
nodes_filtered_df, edges_filtered_df = adapt_to_cytoscape(nodes_filtered_df, edges_filtered_df)


Adding 21613 missing nodes


In [17]:
### get connectivity score
import pandas as pd
import numpy as np

def add_shared_hit_connectivity_score(
    nodes_df,
    edges_df,
    source_col="source",
    target_col="target",
    hits_col="all_hits",
    new_col="connected_multi_degree_hits"
):
    """
    For each query/source node (nodes with numeric all_hits > 0),
    count how many connected target nodes have degree > 1.
    Adds a new column to nodes_df:
        connected_multi_degree_hits
    Parameters
    ----------
    nodes_df : pd.DataFrame
        Node table indexed by node id.
    edges_df : pd.DataFrame
        Edge table with source/target columns.
    Returns
    -------
    nodes_out : pd.DataFrame
        Copy of nodes_df with new metric column added.
    """
    nodes_out = nodes_df.copy()
    # ---------------------------------------------------------
    # identify query/source nodes
    # ---------------------------------------------------------
    query_mask = (
        pd.to_numeric(nodes_out[hits_col], errors="coerce") > 0
    )
    query_nodes = set(nodes_out.index[query_mask])
    # ---------------------------------------------------------
    # calculate node degree from edge table
    # ---------------------------------------------------------
    all_edge_nodes = pd.concat([
        edges_df[source_col],
        edges_df[target_col]
    ])
    degree_series = all_edge_nodes.value_counts()
    # nodes with degree > 1
    multi_degree_nodes = set(
        degree_series[degree_series > 1].index
    )
    # ---------------------------------------------------------
    # for each query node:
    # count connected neighbors with degree > 1
    # ---------------------------------------------------------
    metric_dict = {}
    for node_id in query_nodes:
        connected_targets = set(
            edges_df.loc[
                edges_df[source_col] == node_id,
                target_col
            ]
        )
        connected_sources = set(
            edges_df.loc[
                edges_df[target_col] == node_id,
                source_col
            ]
        )
        neighbors = connected_targets.union(connected_sources)
        shared_neighbors = neighbors.intersection(multi_degree_nodes)
        metric_dict[node_id] = len(shared_neighbors)
    # ---------------------------------------------------------
    # add metric column
    # ---------------------------------------------------------
    nodes_out[new_col] = (
        nodes_out.index.map(metric_dict)
        .fillna(0)
        .astype(int)
    )
    return nodes_out

In [18]:
### apply function using filtered dfs
nodes_scored_df = add_shared_hit_connectivity_score(
    nodes_filtered_df,
    edges_filtered_df
)

### save to file
filtered_scored_table = os.path.join(outdir, os.path.basename(hogs_metadata_file).replace(".tsv","_filteredscored.tsv"))
print(f"Saving to: {filtered_scored_table}")
nodes_scored_df.to_csv(filtered_scored_table, sep='\t')

nodes_scored_df.head()

Saving to: /home/agavriil/Documents/venom_project/A_venom_analysis_tidy/3_results/2_fastoma_results/levels_fastoma_metazoa_251114_subhogs_eventlim1_260619_venomexpression_blastphits/venomhog_full_subhog_metadata_summary_filteredscored.tsv


,is_toxin,all_hits,direct_hit,all_hits_expr,direct_hit_expr,level_taxon,total_species,venomous_species_percent,completeness_score,hashid,is_roothog,Protein families,nodetype,exprmods,id,connected_multi_degree_hits
0_HOG:0117789_1,0.0,5.0,0.0,0.0,0.0,Metazoa,330.0,0.118182,0.9621,352807.0,True,Glutaminyl-peptide cyclotransferase family,venom,,0_HOG:0117789_1,0
0_HOG:0134318_1,0.0,3.0,1.0,0.0,0.0,Metazoa,323.0,0.136223,0.9417,257132.0,True,Apyrase family,venom,,0_HOG:0134318_1,0
0_HOG:0139986_1,0.0,1.0,0.0,0.0,0.0,Metazoa,328.0,0.131098,0.9563,365965.0,True,"Damage-control phosphatase family, Sugar phosp...",venom,,0_HOG:0139986_1,0
101_HOG:0027796.1b_15,0.0,4.0,0.0,4.0,2.0,Amniota,113.0,0.079646,0.9912,82488.0,False,5'-nucleotidase family,venom,"mod24,mod40",101_HOG:0027796.1b_15,0
101_HOG:0035784_15,0.0,37.0,0.0,0.0,0.0,Amniota,108.0,0.083333,0.9474,87835.0,False,NGF-beta family,venom,,101_HOG:0035784_15,33


In [19]:
### do it again but for filtered

### deleting existing networks to start over - this took 2 mins for 180000 edges and 65000 nodes
p4c.delete_all_networks()
collection_title = "HogProf results per taxon"
edges_taxon_col = 'taxname'
nodes_taxon_col = 'level_taxon'
### separate the edges table into a list of dfs, per taxon_name of query
taxa_edges_dfs_list = []

for taxon_name in nodes_scored_df[nodes_taxon_col].unique():
    taxon_nodes_df = nodes_scored_df[nodes_scored_df[nodes_taxon_col] == taxon_name]
    taxon_edges_df = edges_filtered_df[edges_filtered_df[edges_taxon_col] == taxon_name]
    print(f"Taxon {taxon_name} has {len(taxon_edges_df)} interactions among {len(taxon_nodes_df)} HOGs.")
    ### filter out possible hits that are in the nodes still - redundant thanks to our earlier check
    #all_nodes_in_edges = set(taxon_edges_df['source']).union(set(taxon_edges_df['target']))
    #taxon_nodes_df = nodes_df[nodes_df['id'].isin(all_nodes_in_edges)]
    #if len(taxon_nodes_df) > 0:
    p4c.create_network_from_data_frames(taxon_nodes_df, taxon_edges_df, title=taxon_name, collection=collection_title)
    p4c.layout_network('force-directed')
    apply_visual_style(style_name) ### using hogprof_custom from the previous block
    #break

    
### give it a second for the style to be applied
time.sleep(1)
### save the figure
today = date.today()
todaystr = today.strftime("%y%m%d")
### save session for the future
outcys_session_name = f'{todaystr}_HogProf_venom_filtered_0.3medium_taxa_network.cys'
print(f"Saving session in file {outcys_session_name}")
p4c.save_session(os.path.join(outdir, outcys_session_name))

Taxon Metazoa has 21 interactions among 24 HOGs.
Applying default style...
Applying preferred layout
Applying visual style: hogprof_venom_expr_custom
Taxon Amniota has 3330 interactions among 1288 HOGs.
Applying default style...
Applying preferred layout
Applying visual style: hogprof_venom_expr_custom
Taxon Clupeocephala has 1140 interactions among 993 HOGs.
Applying default style...
Applying preferred layout
Applying visual style: hogprof_venom_expr_custom
Taxon Hexapoda has 150 interactions among 155 HOGs.
Applying default style...
Applying preferred layout
Applying visual style: hogprof_venom_expr_custom
Taxon Acari has 28 interactions among 30 HOGs.
Applying default style...
Applying preferred layout
Applying visual style: hogprof_venom_expr_custom
Taxon Chordata has 674 interactions among 616 HOGs.
Applying default style...
Applying preferred layout
Applying visual style: hogprof_venom_expr_custom
Taxon Mammalia has 229 interactions among 233 HOGs.
Applying default style...
Apply

{}

### 1a Taxon-based venom coevolution
Get some metrics on venom coevolution in different taxa

In [20]:
taxon_venom_coevolution_dict = {}
for taxon_name in nodes_df[nodes_taxon_col].unique():
    taxon_nodes_df = nodes_df[nodes_df[nodes_taxon_col] == taxon_name]
    taxon_edges_df = edges_df[edges_df[edges_taxon_col] == taxon_name]
    #print(f"Taxon {taxon_name} has {len(taxon_edges_df)} interactions among {len(taxon_nodes_df)} HOGs.")
    ### filter out possible hits that are in the nodes still - redundant thanks to our earlier check
    #all_nodes_in_edges = set(taxon_edges_df['source']).union(set(taxon_edges_df['target']))
    #taxon_nodes_df = nodes_df[nodes_df['id'].isin(all_nodes_in_edges)]
    taxon_nodes_df, taxon_edges_df = adapt_to_cytoscape(taxon_nodes_df, taxon_edges_df)
    if len(taxon_nodes_df) > 0:
        print(f"Taxon {taxon_name} has {len(taxon_edges_df)} interactions among {len(taxon_nodes_df)} HOGs.\n")
        ### get this taxon's venom hogs 
        all_venom_hogs_list = taxon_nodes_df[taxon_nodes_df['is_toxin'].isin([0,1])]['id'].to_list()

        ### check if both source and target are in the venom hogs list
        venom_edges_len = len(taxon_edges_df[taxon_edges_df['source'].isin(all_venom_hogs_list) & taxon_edges_df['target'].isin(all_venom_hogs_list)])
        taxon_venom_coevolution_dict[taxon_name] = {
            "all_edges": len(taxon_edges_df),
            "venom_edges": venom_edges_len,
            "venom_hogs": len(all_venom_hogs_list)}
        #break

### turn dict into a df
taxon_venom_coevolution_df = pd.DataFrame.from_dict(taxon_venom_coevolution_dict, orient='index')
taxon_venom_coevolution_df = taxon_venom_coevolution_df.sort_values(by='all_edges', ascending=False)
taxon_venom_coevolution_df

/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index
/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index
/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

Taxon Metazoa has 21 interactions among 24 HOGs.

Taxon Amniota has 3330 interactions among 1288 HOGs.

Taxon Clupeocephala has 1140 interactions among 993 HOGs.

Taxon Hexapoda has 150 interactions among 155 HOGs.

Taxon Acari has 28 interactions among 30 HOGs.

Taxon Chordata has 674 interactions among 616 HOGs.

Taxon Mammalia has 229 interactions among 233 HOGs.

Taxon Sauria has 2483 interactions among 1247 HOGs.

Taxon Euteleosteomorpha has 1024 interactions among 1029 HOGs.

Taxon Otophysi has 294 interactions among 295 HOGs.

Taxon Neoptera has 170 interactions among 175 HOGs.

Taxon Ecdysozoa has 343 interactions among 344 HOGs.

Taxon Theria has 245 interactions among 248 HOGs.

Taxon Lophotrochozoa has 259 interactions among 261 HOGs.



/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index
/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index
/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

Taxon Lepidosauria has 2547 interactions among 1622 HOGs.

Taxon Archelosauria has 89 interactions among 92 HOGs.

Taxon Acanthomorphata has 58 interactions among 61 HOGs.

Taxon Endopterygota has 609 interactions among 612 HOGs.

Taxon Eutheria has 506 interactions among 464 HOGs.

Taxon Episquamata has 2715 interactions among 1554 HOGs.

Taxon Gnathostomata has 1656 interactions among 996 HOGs.

Taxon Archosauria has 32 interactions among 34 HOGs.

Taxon Percomorphaceae has 77 interactions among 80 HOGs.

Taxon Hymenoptera has 36 interactions among 42 HOGs.

Taxon Diptera has 215 interactions among 217 HOGs.

Taxon Eumetazoa has 67 interactions among 59 HOGs.

Taxon Boreoeutheria has 191 interactions among 186 HOGs.



/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index
/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index
/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

Taxon Toxicofera has 1338 interactions among 1074 HOGs.

Taxon Neognathae has 9 interactions among 10 HOGs.

Taxon Panarthropoda has 341 interactions among 322 HOGs.

Taxon Ovalentaria has 2 interactions among 4 HOGs.

Taxon Nematoda has 2 interactions among 3 HOGs.

Taxon Apocrita has 38 interactions among 44 HOGs.

Taxon Brachycera has 171 interactions among 173 HOGs.

Taxon Euarchontoglires has 197 interactions among 199 HOGs.

Taxon Laurasiatheria has 63 interactions among 65 HOGs.

Taxon Mollusca has 35 interactions among 36 HOGs.

Taxon Telluraves has 3 interactions among 4 HOGs.

Taxon Atherinomorphae has 88 interactions among 89 HOGs.

Taxon Aculeata has 9 interactions among 11 HOGs.

Taxon Schizophora has 185 interactions among 186 HOGs.

Taxon Primates has 32 interactions among 34 HOGs.

Taxon Euteleostomi has 2590 interactions among 1234 HOGs.

Taxon Haplorrhini has 59 interactions among 61 HOGs.

Taxon Arthropoda has 506 interactions among 456 HOGs.

Taxon Rhabditida has 1 

/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index
/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nodes_df["id"] = nodes_df.index
/tmp/ipykernel_12324/2738804344.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

,all_edges,venom_edges,venom_hogs
Amniota,3330,13,13
Episquamata,2715,7,15
Euteleostomi,2590,7,18
Lepidosauria,2547,8,16
Sauria,2483,6,11
...,...,...,...
Nematoda,2,0,1
Ovalentaria,2,0,2
Drosophila,1,0,1
Rhabditomorpha,1,0,1


In [21]:
taxon_venom_coevolution_df['venom_edges_fraction'] = taxon_venom_coevolution_df['venom_edges'] / taxon_venom_coevolution_df['all_edges']
taxon_venom_coevolution_df[taxon_venom_coevolution_df['venom_edges']>0]
taxon_venom_coevolution_df["max_venom_edges"] = taxon_venom_coevolution_df["venom_hogs"] * (taxon_venom_coevolution_df["venom_hogs"] - 1) / 2
taxon_venom_coevolution_df["venom_edges_fraction_of_max"] = taxon_venom_coevolution_df["venom_edges"] / taxon_venom_coevolution_df["max_venom_edges"]
taxon_venom_coevolution_df["venom_edges_fraction_of_max"] = taxon_venom_coevolution_df["venom_edges_fraction_of_max"].fillna(0)
taxon_venom_coevolution_df.sort_values(by=['venom_edges_fraction_of_max','max_venom_edges'], ascending=False, inplace=True)
venom_coevolution_summary_file = os.path.join(outdir, f"venom_coevolution_taxonsummary.tsv")
taxon_venom_coevolution_df.to_csv(venom_coevolution_summary_file, sep='\t')
print(f"Saved venom coevolution summary to {venom_coevolution_summary_file}")
taxon_venom_coevolution_df[taxon_venom_coevolution_df['venom_edges']>0]

Saved venom coevolution summary to /home/agavriil/Documents/venom_project/A_venom_analysis_tidy/3_results/2_fastoma_results/levels_fastoma_metazoa_251114_subhogs_eventlim1_260619_venomexpression_blastphits/venom_coevolution_taxonsummary.tsv


,all_edges,venom_edges,venom_hogs,venom_edges_fraction,max_venom_edges,venom_edges_fraction_of_max
Amniota,3330,13,13,0.003904,78.0,0.166667
Sauria,2483,6,11,0.002416,55.0,0.109091
Arthropoda,506,2,7,0.003953,21.0,0.095238
Sarcopterygii,2220,7,14,0.003153,91.0,0.076923
Lepidosauria,2547,8,16,0.003141,120.0,0.066667
Episquamata,2715,7,15,0.002578,105.0,0.066667
Tetrapoda,1953,5,14,0.002560,91.0,0.054945
Toxicofera,1338,6,16,0.004484,120.0,0.050000
Euteleostomi,2590,7,18,0.002703,153.0,0.045752
Gnathostomata,1656,2,18,0.001208,153.0,0.013072
